# 02. Automated Financial Data Acquisition via SEC EDGAR API
## Corporate Carbon Footprint Tracker (Task EDA-1)
### Deliverable 1: Data Collection, Extraction, Cleaning & Documentation

---

### Objective
Retrieve genuine audited annual financial revenues for the target 12-company cohort over **FY2019–FY2023** using the official **U.S. Securities and Exchange Commission (SEC) EDGAR API**.

### Technical Protocol & Fair Access Compliance
- **API Endpoint**: `https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json`
- **SEC Fair Access Policy**:
  - Requires declared `User-Agent` format: `User-Agent: ProjectName ContactEmail`.
  - Rate-limited to max 10 requests/second. We enforce a 0.2s pause between calls.
- **Data Immortality**:
  - Full raw JSON responses are saved to `data/raw/sec_edgar/` as immutable audit trails.
- **US-GAAP XBRL Taxonomy Extraction**:
  - Extracts 10-K audited annual revenues (`form == '10-K'` and `fp == 'FY'`).
  - Implements multi-tag fallback priority:
    1. `RevenueFromContractWithCustomerExcludingAssessedTax`
    2. `Revenues`
    3. `SalesRevenueNet`
  - Restatement Resolution: When multiple 10-K filings disclose a fiscal year, the latest filed accession is retained.


In [1]:
import os
import json
import time
import requests
import pandas as pd
import numpy as np
from datetime import datetime

# Directory paths
RAW_SEC_DIR = os.path.join("..", "data", "raw", "sec_edgar")
INVENTORY_CSV = os.path.join("..", "data", "raw", "source_inventory.csv")
OUT_CSV = os.path.join("..", "data", "interim", "raw_sec_financials.csv")

os.makedirs(RAW_SEC_DIR, exist_ok=True)
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)

print("Environment setup complete.")


Environment setup complete.


## 1. Load Source Inventory & Target CIKs

In [2]:
df_inv = pd.read_csv(INVENTORY_CSV)
print(f"Loaded {len(df_inv)} companies.")
df_inv[['company_id', 'company_name', 'ticker', 'cik']]


Loaded 22 companies.


,company_id,company_name,ticker,cik
0,TECH_MSFT,Microsoft Corporation,MSFT,789019
1,TECH_AAPL,Apple Inc.,AAPL,320193
2,TECH_GOOGL,Alphabet Inc.,GOOGL,1652044
3,TECH_AMZN,Amazon.com Inc.,AMZN,1018724
4,TECH_META,Meta Platforms Inc.,META,1326801
5,TECH_INTC,Intel Corporation,INTC,50863
6,TECH_NVDA,NVIDIA Corporation,NVDA,1045810
7,AUTO_GM,General Motors Company,GM,1467858
8,ENG_XOM,ExxonMobil Corporation,XOM,34088
9,ENG_CVX,Chevron Corporation,CVX,93410


## 2. SEC EDGAR API Data Retrieval Pipeline
Iterate through all registered companies, check local cache, and query the SEC REST endpoint.


In [3]:
headers = {
    'User-Agent': 'CorporateCarbonFootprintTracker BDAProject/1.0 (academic_research@pes.edu)'
}

for _, row in df_inv.iterrows():
    ticker = row['ticker']
    cik = str(row['cik']).zfill(10)
    json_path = os.path.join(RAW_SEC_DIR, f"CIK{cik}.json")
    
    if not os.path.exists(json_path) or os.path.getsize(json_path) < 1000:
        url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
        print(f"Querying SEC EDGAR for {ticker} (CIK {cik})...")
        try:
            resp = requests.get(url, headers=headers, timeout=15)
            if resp.status_code == 200:
                with open(json_path, "w", encoding="utf-8") as f:
                    f.write(resp.text)
                print(f"  -> Successfully saved {len(resp.text):,} bytes to {os.path.basename(json_path)}")
            else:
                print(f"  -> HTTP Error {resp.status_code}")
        except Exception as e:
            print(f"  -> Request Exception: {e}")
        time.sleep(0.2)
    else:
        print(f"Verified cached SEC disclosure for {ticker} ({os.path.getsize(json_path):,} bytes).")


Verified cached SEC disclosure for MSFT (4,881,196 bytes).
Verified cached SEC disclosure for AAPL (3,789,099 bytes).
Verified cached SEC disclosure for GOOGL (3,164,517 bytes).
Verified cached SEC disclosure for AMZN (4,478,611 bytes).
Verified cached SEC disclosure for META (2,746,680 bytes).
Verified cached SEC disclosure for INTC (4,311,809 bytes).
Verified cached SEC disclosure for NVDA (4,092,966 bytes).
Verified cached SEC disclosure for GM (3,606,453 bytes).
Verified cached SEC disclosure for XOM (3,122,723 bytes).
Verified cached SEC disclosure for CVX (4,810,683 bytes).
Verified cached SEC disclosure for KO (5,028,478 bytes).
Verified cached SEC disclosure for JNJ (4,168,526 bytes).
Verified cached SEC disclosure for LMT (3,380,288 bytes).
Verified cached SEC disclosure for HPQ (4,957,401 bytes).
Verified cached SEC disclosure for F (3,672,318 bytes).
Querying SEC EDGAR for TSLA (CIK 0001318605)...
  -> Successfully saved 3,647,444 bytes to CIK0001318605.json
Verified cached 

## 3. Parse XBRL Financial Facts & Extract Annual 10-K Revenues

In [4]:
results = []

revenue_tags_priority = [
    'RevenueFromContractWithCustomerExcludingAssessedTax',
    'Revenues',
    'SalesRevenueNet'
]

target_years = [2019, 2020, 2021, 2022, 2023]

for _, row in df_inv.iterrows():
    cid = row['company_id']
    cname = row['company_name']
    ticker = row['ticker']
    cik = str(row['cik']).zfill(10)
    json_path = os.path.join(RAW_SEC_DIR, f"CIK{cik}.json")
    
    with open(json_path, "r", encoding="utf-8") as f:
        sec_data = json.load(f)
        
    us_gaap = sec_data.get('facts', {}).get('us-gaap', {})
    
    for fy in target_years:
        found_entry = None
        found_tag = None
        
        for tag in revenue_tags_priority:
            if tag in us_gaap and 'USD' in us_gaap[tag].get('units', {}):
                candidates = [
                    u for u in us_gaap[tag]['units']['USD']
                    if u.get('form') == '10-K' and u.get('fp') == 'FY' and u.get('fy') == fy
                ]
                if candidates:
                    # Prefer full-year (>300 days) observations
                    fy_cands = []
                    for u in candidates:
                        if 'start' in u and 'end' in u:
                            try:
                                d = (datetime.strptime(u['end'], '%Y-%m-%d') - datetime.strptime(u['start'], '%Y-%m-%d')).days
                                if d > 300:
                                    fy_cands.append(u)
                            except:
                                pass
                    if fy_cands:
                        candidates = fy_cands
                    candidates.sort(key=lambda x: x.get('filed', ''))
                    found_entry = candidates[-1]
                    found_tag = tag
                    break
                    
        if found_entry:
            val = found_entry.get('val')
            results.append({
                'company_id': cid,
                'company_name': cname,
                'ticker': ticker,
                'cik': cik,
                'reporting_year': fy,
                'revenue_value': val,
                'revenue_currency': 'USD',
                'revenue_usd_m': round(val / 1e6, 2) if val else None,
                'sec_accession_number': found_entry.get('accn'),
                'sec_filing_date': found_entry.get('filed'),
                'xbrl_tag': found_tag
            })

df_financials = pd.DataFrame(results)
print(f"Total Extracted Company-Year Records: {len(df_financials)} / {len(df_inv) * len(target_years)}")
df_financials.head(15)


Total Extracted Company-Year Records: 110 / 110


,company_id,company_name,ticker,cik,reporting_year,revenue_value,revenue_currency,revenue_usd_m,sec_accession_number,sec_filing_date,xbrl_tag
0,TECH_MSFT,Microsoft Corporation,MSFT,0000789019,2019,125843000000,USD,125843.0,0001564590-19-027952,2019-08-01,RevenueFromContractWithCustomerExcludingAssess...
1,TECH_MSFT,Microsoft Corporation,MSFT,0000789019,2020,143015000000,USD,143015.0,0001564590-20-034944,2020-07-30,RevenueFromContractWithCustomerExcludingAssess...
2,TECH_MSFT,Microsoft Corporation,MSFT,0000789019,2021,168088000000,USD,168088.0,0001564590-21-039151,2021-07-29,RevenueFromContractWithCustomerExcludingAssess...
3,TECH_MSFT,Microsoft Corporation,MSFT,0000789019,2022,198270000000,USD,198270.0,0001564590-22-026876,2022-07-28,RevenueFromContractWithCustomerExcludingAssess...
4,TECH_MSFT,Microsoft Corporation,MSFT,0000789019,2023,211915000000,USD,211915.0,0000950170-23-035122,2023-07-27,RevenueFromContractWithCustomerExcludingAssess...
5,TECH_AAPL,Apple Inc.,AAPL,0000320193,2019,260174000000,USD,260174.0,0000320193-19-000119,2019-10-31,RevenueFromContractWithCustomerExcludingAssess...
6,TECH_AAPL,Apple Inc.,AAPL,0000320193,2020,274515000000,USD,274515.0,0000320193-20-000096,2020-10-30,RevenueFromContractWithCustomerExcludingAssess...
7,TECH_AAPL,Apple Inc.,AAPL,0000320193,2021,365817000000,USD,365817.0,0000320193-21-000105,2021-10-29,RevenueFromContractWithCustomerExcludingAssess...
8,TECH_AAPL,Apple Inc.,AAPL,0000320193,2022,394328000000,USD,394328.0,0000320193-22-000108,2022-10-28,RevenueFromContractWithCustomerExcludingAssess...
9,TECH_AAPL,Apple Inc.,AAPL,0000320193,2023,383285000000,USD,383285.0,0000320193-23-000106,2023-11-03,RevenueFromContractWithCustomerExcludingAssess...


## 4. Save Interim Dataset & Verify Completeness

In [5]:
df_financials.to_csv(OUT_CSV, index=False)
print(f"Interim financial dataset exported to: {OUT_CSV}")

# Pivot summary to confirm zero missingness
pivot_rev = df_financials.pivot(index=['company_id', 'ticker'], columns='reporting_year', values='revenue_usd_m')
display(pivot_rev)

print("\nRevenue Summary Statistics (in USD Millions):")
display(df_financials.groupby('ticker')['revenue_usd_m'].describe()[['min', 'mean', 'max']])


Interim financial dataset exported to: ..\data\interim\raw_sec_financials.csv


,reporting_year,2019,2020,2021,2022,2023
company_id,ticker,,,,,
AUTO_F,F,155900.0,127144.0,136341.0,158057.0,176191.0
AUTO_GM,GM,137237.0,108673.0,113590.0,143975.0,157658.0
AUTO_TSLA,TSLA,24578.0,31536.0,53823.0,81462.0,96773.0
CONS_KO,KO,37266.0,33014.0,38655.0,43004.0,45754.0
CONS_NKE,NKE,39117.0,37403.0,44538.0,46710.0,51217.0
ENG_COP,COP,32567.0,18784.0,45828.0,61049.0,48522.0
ENG_CVX,CVX,139865.0,94471.0,155606.0,235717.0,196913.0
ENG_XOM,XOM,255583.0,178574.0,276692.0,413680.0,344582.0
HLTH_JNJ,JNJ,82059.0,82584.0,93775.0,94943.0,85159.0



Revenue Summary Statistics (in USD Millions):


,min,mean,max
ticker,,,
AAPL,260174.0,335623.8,394328.0
AMZN,280522.0,445035.2,574785.0
CAT,41748.0,54601.2,67060.0
COP,18784.0,41350.0,61049.0
CVX,94471.0,164514.4,235717.0
F,127144.0,150726.6,176191.0
GM,108673.0,132226.6,157658.0
GOOGL,161857.0,238450.2,307394.0
HPQ,53718.0,59116.6,63487.0


## 5. Next Steps
The official audited revenues for all 60 company-year pairs have been verified and exported to `data/interim/raw_sec_financials.csv`.
Proceed to **`03_pdf_extraction.ipynb`** to download corporate sustainability reports and extract Scope 1, Scope 2, and Scope 3 greenhouse gas emissions.
